<a href="https://colab.research.google.com/github/nilusrubanathan-byte/MSC-AI-University-of-Essex/blob/main/Assignment_2_2_a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

Below is a complete, self‑contained package you can use for Assignment 2, fully aligned with your first assignment and including Shannon entropy as an optional enhancement. You can:
* Use the single‑file version directly in Google Colab.
* Use the module structure if you want to split it into multiple files locally.

All content is in British English.

## Implementation Plan (aligned with Assignment 1)

**Goal:** Implement a reactive digital forensics agent that:
* Scans directories (BFS + generators).
* Identifies relevant files using extension + magic number.
* Computes SHA‑256 + MD5 hashes.
* Optionally computes Shannon entropy as supplementary metadata.
* Archives evidence (TAR/GZ).
* Stores metadata in SQLite.
* Logs custody events (HMAC‑signed).
* Handles permission errors gracefully.

**Key design commitments from Assignment 1:**
* Single reactive agent, not BDI or multi‑agent.
* Two‑stage identification strategy: extension filter + magic number.
* Generator‑based BFS for scalability.
* Lightweight SQLite for metadata and audit logs.
* Read‑only by default, with minimal attack surface.

Now, to proceed with executing and explaining the code step-by-step, I need to see the content of the file `/content/Executable codes Assig 2.txt`. The following cell will read and display its content.

In [1]:
with open('/content/Executable codes Assig 2.txt', 'r') as f:
    code_content = f.read()

print(code_content)

2. Python module structure (if you split into files)
If you are working locally (not in Colab), you can structure it like this:

forensic_agent/
│
├─ config.py
├─ scanner.py
├─ identifier.py
├─ hasher.py
├─ entropy.py
├─ db.py
├─ archiver.py
├─ custody.py
├─ access_control.py
├─ agent.py
└─ tests/
    ├─ test_scanner.py
    ├─ test_identifier.py
    ├─ test_hasher.py
    ├─ test_entropy.py
    └─ test_db.py

3. Full executable code (single file for Google Colab)
You can copy and paste this entire block into a Colab cell and run it. It includes:
• 	Configuration
• 	Entropy (Shannon)
• 	Scanner (BFS)
• 	Identifier (extension + magic number)
• 	Hasher (SHA‑256 + MD5)
• 	SQLite metadata store
• 	Archive manager (TAR/GZ)
• 	Custody logger (HMAC)
• 	Access control
• 	Forensic agent orchestration
• 	Simple test/demo runner

# ==========================
# CONFIGURATION
# ==========================
import os
from pathlib import Path

class Config:
    # Root directory to scan (change this in Co

## 1. Python Module Structure (for local development)

This section outlines a recommended directory and file structure if you were to develop this forensics agent as a modular application locally, rather than as a single script in Google Colab. It helps in organizing code into logical units, making it easier to manage and test.

In [2]:
# The content below describes a suggested module structure, not executable code.
module_structure_description = '''
forensic_agent/
│
├─ config.py
├─ scanner.py
├─ identifier.py
├─ hasher.py
├─ entropy.py
├─ db.py
├─ archiver.py
├─ custody.py
├─ access_control.py
├─ agent.py
└─ tests/
    ├─ test_scanner.py
    ├─ test_identifier.py
    ├─ test_hasher.py
    ├─ test_entropy.py
    └─ test_db.py
'''
print(module_structure_description)


forensic_agent/
│
├─ config.py
├─ scanner.py
├─ identifier.py
├─ hasher.py
├─ entropy.py
├─ db.py
├─ archiver.py
├─ custody.py
├─ access_control.py
├─ agent.py
└─ tests/
    ├─ test_scanner.py
    ├─ test_identifier.py
    ├─ test_hasher.py
    ├─ test_entropy.py
    └─ test_db.py



### 2.4. FILE IDENTIFIER (EXTENSION + MAGIC NUMBER)

This section defines the `FileIdentifier` class, which is responsible for determining if a file is relevant for forensic analysis. It uses a two-stage approach: first checking the file's extension, and then, if a `python-magic` library is available, verifying its MIME type using magic numbers. Magic numbers are bytes within a file that identify its format, providing a more robust check than just the extension. Note the explicit import of `from pathlib import Path` to resolve dependencies within the class definition.

In [6]:
from pathlib import Path # Added this import

try:
    import magic  # python-magic
    MAGIC_AVAILABLE = True
except ImportError:
    MAGIC_AVAILABLE = False

class FileIdentifier:
    def __init__(self, target_extensions, mime_map):
        self.target_extensions = target_extensions
        self.mime_map = mime_map
        if MAGIC_AVAILABLE:
            # Initialise libmagic to check MIME types
            self.magic = magic.Magic(mime=True)
        else:
            self.magic = None

    def is_relevant(self, path: Path) -> bool:
        """
        Two-stage identification:
        1. Extension filter
        2. Magic number / MIME verification (if python-magic is available)
        """
        # Stage 1: Extension filter
        if path.suffix.lower() not in self.target_extensions:
            return False

        # Stage 2: Magic number / MIME verification (if available)
        if self.magic:
            try:
                # Get the actual MIME type using libmagic
                actual_mime = self.magic.from_file(str(path))
                # Check if the actual MIME type matches expected types for its extension
                expected_mimes = self.mime_map.get(path.suffix.lower(), [])
                return actual_mime in expected_mimes
            except Exception as e:
                # Handle potential errors during magic number check (e.g., file unreadable)
                print(f"Warning: Could not check magic number for {path}: {e}")
                return False # Treat as not relevant if check fails
        else:
            # If python-magic is not available, rely solely on extension
            return True

### Test Case for `create_final_archive`

This test will:
1. Create a temporary directory and some dummy `.txt` files within it.
2. Instantiate the `ArchiveManager`.
3. Manually add these dummy files to the `archive_manager`'s internal list.
4. Call `create_final_archive()` to create the `.tar.gz` archive.
5. Verify that the archive file exists and contains the expected files.
6. Clean up the temporary files and the archive.

In [25]:
import os
import tarfile
from pathlib import Path
import shutil

# Assuming ArchiveManager class is defined in the current scope (from cell cb107983)

# 1. Setup: Create a temporary directory and dummy files
test_dir = Path("./temp_archive_test")
test_dir.mkdir(exist_ok=True)

dummy_files = []
for i in range(3):
    file_path = test_dir / f"test_file_{i}.txt"
    file_path.write_text(f"This is content for test file {i}.")
    dummy_files.append(file_path)

archive_output_path = Path("test_archive.tar.gz")

print(f"Created temporary directory: {test_dir}")
print(f"Created dummy files: {dummy_files}")

# 2. Instantiate ArchiveManager
archive_manager = ArchiveManager(archive_output_path)

# 3. Manually add files to the internal list
for f_path in dummy_files:
    archive_manager.add_to_archive(f_path)

print(f"Files marked for archiving: {archive_manager.files_to_archive}")

# 4. Call create_final_archive()
print("Calling create_final_archive...")
success = archive_manager.create_final_archive()

# 5. Verification
if success and archive_output_path.exists():
    print(f"Archive '{archive_output_path}' created successfully.")
    try:
        with tarfile.open(archive_output_path, 'r:gz') as tar:
            archived_members = [Path(m.name) for m in tar.getmembers() if m.isfile()]

            # Compare relative paths for verification
            expected_archived_names = [f.relative_to(Path.cwd()) for f in dummy_files]

            print(f"Expected members in archive: {expected_archived_names}")
            print(f"Actual members in archive: {archived_members}")

            # Check if all expected files are present in the archive
            if all(name in archived_members for name in expected_archived_names):
                print("Test PASSED: All dummy files found in the archive.")
            else:
                print("Test FAILED: Not all dummy files found in the archive.")

    except Exception as e:
        print(f"Test FAILED: Error inspecting archive: {e}")
else:
    print(f"Test FAILED: Archive '{archive_output_path}' was not created or creation failed.")

# 6. Cleanup
print("Cleaning up temporary files and archive...")
if test_dir.exists():
    shutil.rmtree(test_dir)
    print(f"Removed directory: {test_dir}")
if archive_output_path.exists():
    os.remove(archive_output_path)
    print(f"Removed archive: {archive_output_path}")

print("Test case finished.")

Created temporary directory: temp_archive_test
Created dummy files: [PosixPath('temp_archive_test/test_file_0.txt'), PosixPath('temp_archive_test/test_file_1.txt'), PosixPath('temp_archive_test/test_file_2.txt')]
Files marked for archiving: [PosixPath('temp_archive_test/test_file_0.txt'), PosixPath('temp_archive_test/test_file_1.txt'), PosixPath('temp_archive_test/test_file_2.txt')]
Calling create_final_archive...
Error creating final archive /content/test_archive.tar.gz: 'temp_archive_test/test_file_0.txt' is not in the subpath of '/content'
Test FAILED: Archive 'test_archive.tar.gz' was not created or creation failed.
Cleaning up temporary files and archive...
Removed directory: temp_archive_test
Removed archive: test_archive.tar.gz
Test case finished.


In [33]:
import os
import tarfile
from pathlib import Path
import shutil

# Assuming ArchiveManager class is defined in the current scope (from cell cb107983)

# 1. Setup: Create a temporary directory and dummy files
test_dir = Path("./temp_archive_test")
test_dir.mkdir(exist_ok=True)

dummy_files = []
for i in range(3):
    file_path = test_dir / f"test_file_{i}.txt"
    file_path.write_text(f"This is content for test file {i}.")
    dummy_files.append(file_path)

archive_output_path = Path("test_archive.tar.gz")

print(f"Created temporary directory: {test_dir}")
print(f"Created dummy files: {dummy_files}")

# 2. Instantiate ArchiveManager
archive_manager = ArchiveManager(archive_output_path)

# 3. Manually add files to the internal list
for f_path in dummy_files:
    archive_manager.add_to_archive(f_path)

print(f"Files marked for archiving: {archive_manager.files_to_archive}")

# 4. Call create_final_archive()
print("Calling create_final_archive...")
success = archive_manager.create_final_archive()

# 5. Verification
if success and archive_output_path.exists():
    print(f"Archive '{archive_output_path}' created successfully.")
    try:
        with tarfile.open(archive_output_path, 'r:gz') as tar:
            archived_members = [Path(m.name) for m in tar.getmembers() if m.isfile()]

            # Compare relative paths for verification
            # The arcname stored in the archive will be the relative path of the file to the original CWD.
            # Since dummy_files are already relative paths from the current execution context (which is /content),
            # we can directly compare them to the archived_members.
            expected_archived_names = dummy_files

            print(f"Expected members in archive: {expected_archived_names}")
            print(f"Actual members in archive: {archived_members}")

            # Check if all expected files are present in the archive
            if all(name in archived_members for name in expected_archived_names):
                print("Test PASSED: All dummy files found in the archive.")
            else:
                print("Test FAILED: Not all dummy files found in the archive.")
                missing_files = [name for name in expected_archived_names if name not in archived_members]
                print(f"Missing files: {missing_files}")

    except Exception as e:
        print(f"Test FAILED: Error inspecting archive: {e}")
else:
    print(f"Test FAILED: Archive '{archive_output_path}' was not created or creation failed.")

# 6. Cleanup
print("Cleaning up temporary files and archive...")
if test_dir.exists():
    shutil.rmtree(test_dir)
    print(f"Removed directory: {test_dir}")
if archive_output_path.exists():
    os.remove(archive_output_path)
    print(f"Removed archive: {archive_output_path}")

print("Test case finished.")

Created temporary directory: temp_archive_test
Created dummy files: [PosixPath('temp_archive_test/test_file_0.txt'), PosixPath('temp_archive_test/test_file_1.txt'), PosixPath('temp_archive_test/test_file_2.txt')]
Files marked for archiving: [PosixPath('temp_archive_test/test_file_0.txt'), PosixPath('temp_archive_test/test_file_1.txt'), PosixPath('temp_archive_test/test_file_2.txt')]
Calling create_final_archive...
Final archive created at: /content/test_archive.tar.gz
Archive 'test_archive.tar.gz' created successfully.
Expected members in archive: [PosixPath('temp_archive_test/test_file_0.txt'), PosixPath('temp_archive_test/test_file_1.txt'), PosixPath('temp_archive_test/test_file_2.txt')]
Actual members in archive: [PosixPath('temp_archive_test/test_file_0.txt'), PosixPath('temp_archive_test/test_file_1.txt'), PosixPath('temp_archive_test/test_file_2.txt')]
Test PASSED: All dummy files found in the archive.
Cleaning up temporary files and archive...
Removed directory: temp_archive_tes

### Inspecting the `evidence.db` Database

First, let's connect to the SQLite database created by the `ForensicAgent` and display the contents of the `evidence` table, which contains the metadata of the identified relevant files.

In [37]:
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect('evidence.db')

# Query the evidence table
evidence_df = pd.read_sql_query("SELECT * FROM evidence", conn)

print("--- Evidence Table Contents ---")
display(evidence_df)

# Query the custody_log table
custody_df = pd.read_sql_query("SELECT * FROM custody_log", conn)

print("\n--- Custody Log Table Contents ---")
display(custody_df)

# Close the connection
conn.close()

--- Evidence Table Contents ---


,id,file_path,file_size,last_modified,sha256_hash,md5_hash,shannon_entropy,timestamp
0,1,Executable codes Assig 2.txt,11467,2026-04-08T15:39:32.716436,fa3ff18f8799560a3425c07271d07a4ac8faf16c70ce48...,cbfbdfe9110bce82fc7477fd32094fb4,4.760033,2026-04-08T17:39:26.680690



--- Custody Log Table Contents ---


,id,timestamp,event_description,hmac_signature
0,1,2026-04-08T17:39:26.667507,Forensic agent initiated scan.,240eb4cfbd010ddf4438b9682b02063dee4308c23bc01e...
1,2,2026-04-08T17:39:26.691457,Metadata recorded for Executable codes Assig 2...,1dc6f987f254a6fb0ec1bf6132b8dbf144973e68adb315...
2,3,2026-04-08T17:39:26.702989,Forensic agent scan completed in 0:00:00.03556...,68a5d70a2f59dd80aea7eeff047e51c6941571f715ad58...
3,4,2026-04-08T19:20:32.683266,Forensic agent initiated scan.,4901d136815c688ed484c3cbe86543c3d798467d2e22ab...
4,5,2026-04-08T19:20:32.695535,Failed to create final evidence archive.,d2704d51bb2880ca875b45bf1bc15bf421768722291ef5...
5,6,2026-04-08T19:20:32.704491,Forensic agent scan completed in 0:00:00.01228...,4b517b4338b9d4b30c8d9666eac28b771a8e494e8f56ec...
6,7,2026-04-08T19:23:46.219612,Forensic agent initiated scan.,eafca90497ea82390e03d30f62a43da29fffb119547386...
7,8,2026-04-08T19:23:46.233119,Failed to create final evidence archive.,58fcdb7aba9c2bc5ef3ab92bc88e42f189a9878c6a2aba...
8,9,2026-04-08T19:23:46.243202,Forensic agent scan completed in 0:00:00.01355...,f3560767ba3679726d5f12c1a4bb80d7ab18a78b2b043b...


### Inspecting the `evidence_archive.tar.gz` Archive

Next, let's verify the contents of the created `evidence_archive.tar.gz` to see which files were actually archived.

# Assignment 2: Digital Forensics Agent Implementation

## Introduction
This document presents a complete, self-contained Python package for Assignment 2, aligned with Assignment 1 requirements. It implements a reactive digital forensics agent capable of scanning directories, identifying relevant files, computing hashes and entropy, archiving evidence, storing metadata in SQLite, and logging custody events with HMAC signatures, all while handling permission errors gracefully. This package is designed to be easily runnable in Google Colab as a single file.

## Implementation Plan (aligned with Assignment 1)
**Goal:** Implement a reactive digital forensics agent that:
*   Scans directories (BFS + generators).
*   Identifies relevant files using extension + magic number.
*   Computes SHA‑256 + MD5 hashes.
*   Optionally computes Shannon entropy as supplementary metadata.
*   Archives evidence (TAR/GZ).
*   Stores metadata in SQLite.
*   Logs custody events (HMAC‑signed).
*   Handles permission errors gracefully.

**Key design commitments from Assignment 1:**
*   Single reactive agent, not BDI or multi‑agent.
*   Two‑stage identification strategy: extension filter + magic number.
*   Generator‑based BFS for scalability.
*   Lightweight SQLite for metadata and audit logs.
*   Read‑only by default, with minimal attack surface.

## Full Executable Code (Single file for Google Colab)
Below is the consolidated Python code containing all class definitions for the forensic agent. This entire block can be copied and pasted into a single Google Colab cell and executed.

```python
import os
import math
import hashlib
import sqlite3
import tarfile
import hmac
from pathlib import Path
from datetime import datetime
from collections import deque

try:
    import magic  # python-magic
    MAGIC_AVAILABLE = True
except ImportError:
    MAGIC_AVAILABLE = False

# ==========================
# CONFIGURATION
# ==========================
class Config:
    ROOT = Path(".")
    DB_PATH = Path("evidence.db")
    ARCHIVE_PATH = Path("evidence_archive.tar.gz")
    CUSTODY_KEY = b"super_secret_key_change_me"
    TARGET_EXTENSIONS = {".txt", ".pdf", ".jpg", ".jpeg", ".png"}
    MIME_MAP = {
        ".txt": ["text/plain"],
        ".pdf": ["application/pdf"],
        ".jpg": ["image/jpeg"],
        ".jpeg": ["image/jpeg"],
        ".png": ["image/png"],
    }

# ==========================
# ENTROPY (SHANNON)
# ==========================
class EntropyCalculator:
    def __init__(self, sample_size: int = 1024 * 64):
        self.sample_size = sample_size

    def file_entropy(self, path: Path) -> float:
        try:
            with path.open("rb") as f:
                data = f.read(self.sample_size)
        except Exception:
            return 0.0

        if not data:
            return 0.0

        freq = [0] * 256
        for b in data:
            freq[b] += 1

        entropy = 0.0
        length = len(data)

        for count in freq:
            if count == 0:
                continue
            p = count / length
            entropy -= p * math.log2(p)

        return entropy

# ==========================
# DIRECTORY SCANNER (BFS + GENERATORS)
# ==========================
class DirectoryScanner:
    def __init__(self, root: Path):
        self.root = Path(root)

    def bfs_scan(self):
        queue = deque([self.root])
        while queue:
            current = queue.popleft()
            if not current.exists():
                continue
            for entry in current.iterdir():
                if entry.is_dir():
                    queue.append(entry)
                else:
                    yield entry

# ==========================
# FILE IDENTIFIER (EXTENSION + MAGIC NUMBER)
# ==========================
class FileIdentifier:
    def __init__(self, target_extensions, mime_map):
        self.target_extensions = target_extensions
        self.mime_map = mime_map
        if MAGIC_AVAILABLE:
            self.magic = magic.Magic(mime=True)
        else:
            self.magic = None

    def is_relevant(self, path: Path) -> bool:
        if path.suffix.lower() not in self.target_extensions:
            return False

        if self.magic:
            try:
                actual_mime = self.magic.from_file(str(path))
                expected_mimes = self.mime_map.get(path.suffix.lower(), [])
                return actual_mime in expected_mimes
            except Exception as e:
                print(f"Warning: Could not check magic number for {path}: {e}")
                return False
        else:
            return True

# ==========================
# HASHER (SHA-256 + MD5)
# ==========================
class Hasher:
    def hash_file(self, path: Path, buffer_size: int = 65536) -> dict:
        sha256_hash = hashlib.sha256()
        md5_hash = hashlib.md5()
        try:
            with open(path, "rb") as f:
                while True:
                    data = f.read(buffer_size)
                    if not data:
                        break
                    sha256_hash.update(data)
                    md5_hash.update(data)
            return {
                "sha256": sha256_hash.hexdigest(),
                "md5": md5_hash.hexdigest()
            }
        except Exception:
            return {"sha256": "ERROR", "md5": "ERROR"}

# ==========================
# SQLITE METADATA STORE
# ==========================
class DatabaseManager:
    def __init__(self, db_path: Path):
        self.db_path = db_path
        self._create_tables()

    def _create_tables(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS evidence (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                file_path TEXT NOT NULL UNIQUE,
                file_size INTEGER,
                last_modified TEXT,
                sha256_hash TEXT,
                md5_hash TEXT,
                shannon_entropy REAL,
                timestamp TEXT NOT NULL
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS custody_log (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT NOT NULL,
                event_description TEXT NOT NULL,
                hmac_signature TEXT NOT NULL
            )
        """)
        conn.commit()
        conn.close()

    def insert_evidence(self, file_path: Path, file_size: int, last_modified: datetime, sha256_hash: str, md5_hash: str, shannon_entropy: float):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        timestamp = datetime.now().isoformat()
        try:
            cursor.execute("""
                INSERT INTO evidence (file_path, file_size, last_modified, sha256_hash, md5_hash, shannon_entropy, timestamp)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (str(file_path), file_size, last_modified.isoformat(), sha256_hash, md5_hash, shannon_entropy, timestamp))
            conn.commit()
            return True
        except sqlite3.IntegrityError:
            print(f"Warning: Evidence for {file_path} already exists. Skipping insertion.")
            return False
        except Exception as e:
            print(f"Error inserting evidence for {file_path}: {e}")
            return False
        finally:
            conn.close()

    def log_custody_event(self, event_description: str, hmac_signature: str):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        timestamp = datetime.now().isoformat()
        try:
            cursor.execute("""
                INSERT INTO custody_log (timestamp, event_description, hmac_signature)
                VALUES (?, ?, ?)
            """, (timestamp, event_description, hmac_signature))
            conn.commit()
            return True
        except Exception as e:
            print(f"Error logging custody event: {e}")
            return False
        finally:
            conn.close()

    def get_all_evidence(self) -> list:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM evidence")
        records = cursor.fetchall()
        conn.close()
        return records

    def get_all_custody_logs(self) -> list:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM custody_log")
        records = cursor.fetchall()
        conn.close()
        return records

# ==========================
# ARCHIVE MANAGER (TAR/GZ)
# ==========================
class ArchiveManager:
    def __init__(self, archive_path: Path):
        self.archive_path = archive_path
        self.files_to_archive = [] # New: list to hold paths of files to be archived

    def add_to_archive(self, file_path: Path):
        self.files_to_archive.append(file_path)
        return True # Indicate that the file is marked for archiving

    def create_final_archive(self):
        if not self.files_to_archive:
            print("No files to archive.")
            return False

        original_cwd = Path.cwd()
        absolute_archive_path = self.archive_path if self.archive_path.is_absolute() else (original_cwd / self.archive_path)

        try:
            with tarfile.open(absolute_archive_path, 'w:gz') as tar:
                for file_path in self.files_to_archive:
                    if file_path.exists():
                        tar.add(str(file_path.resolve()), arcname=os.path.relpath(str(file_path.resolve()), str(original_cwd)))
                    else:
                        print(f"Warning: File not found for archiving: {file_path}")
            print(f"Final archive created at: {absolute_archive_path}")
            return True
        except Exception as e:
            print(f"Error creating final archive {absolute_archive_path}: {e}")
            return False

    def extract_archive(self, extract_path: Path):
        try:
            with tarfile.open(self.archive_path, 'r:gz') as tar:
                tar.extractall(path=extract_path)
            print(f"Archive extracted to {extract_path}")
            return True
        except Exception as e:
            print(f"Error extracting archive: {e}")
            return False

# ==========================
# CUSTODY LOGGER (HMAC)
# ==========================
class CustodyLogger:
    def __init__(self, secret_key: bytes):
        self.secret_key = secret_key

    def create_hmac_signature(self, data: str) -> str:
        h = hmac.new(self.secret_key, data.encode('utf-8'), hashlib.sha256)
        return h.hexdigest()

    def verify_hmac_signature(self, data: str, signature: str) -> bool:
        expected_signature = self.create_hmac_signature(data)
        return hmac.compare_digest(expected_signature, signature)

    def log_event(self, event_description: str) -> dict:
        timestamp = datetime.now().isoformat()
        log_data = f"{timestamp}|{event_description}"
        signature = self.create_hmac_signature(log_data)
        return {
            "timestamp": timestamp,
            "event_description": event_description,
            "hmac_signature": signature,
            "log_data_for_verification": log_data
        }

# ==========================
# ACCESS CONTROL
# ==========================
class AccessControl:
    def __init__(self):
        pass

    def has_read_access(self, path: Path) -> bool:
        try:
            return os.access(path, os.R_OK)
        except Exception as e:
            print(f"Warning: Could not check read access for {path}: {e}")
            return False

    def handle_permission_error(self, path: Path, action: str):
        print(f"Permission Error: Cannot {action} {path}. Skipping.")

# ==========================
# FORENSIC AGENT ORCHESTRATION
# ==========================
class ForensicAgent:
    def __init__(self, config: 'Config'):
        self.config = config
        self.scanner = DirectoryScanner(config.ROOT)
        self.identifier = FileIdentifier(config.TARGET_EXTENSIONS, config.MIME_MAP)
        self.hasher = Hasher()
        self.entropy_calculator = EntropyCalculator()
        self.db_manager = DatabaseManager(config.DB_PATH)
        self.archive_manager = ArchiveManager(config.ARCHIVE_PATH)
        self.custody_logger = CustodyLogger(config.CUSTODY_KEY)
        self.access_control = AccessControl()
        self.start_timestamp = datetime.now()

        log_entry = self.custody_logger.log_event("Forensic agent initiated scan.")
        self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
        print(f"Forensic Agent initialised. Scanning from: {self.config.ROOT}")

    def run_scan(self):
        print("Starting forensic scan...")
        for file_path in self.scanner.bfs_scan():
            if not self.access_control.has_read_access(file_path):
                self.access_control.handle_permission_error(file_path, "read")
                log_entry = self.custody_logger.log_event(f"Permission denied for file: {file_path}. Skipped.")
                self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
                continue

            if self.identifier.is_relevant(file_path):
                print(f"Found relevant file: {file_path}")
                try:
                    file_stat = file_path.stat()
                    file_size = file_stat.st_size
                    last_modified = datetime.fromtimestamp(file_stat.st_mtime)

                    hashes = self.hasher.hash_file(file_path)
                    sha256_hash = hashes['sha256']
                    md5_hash = hashes['md5']

                    shannon_entropy = self.entropy_calculator.file_entropy(file_path)

                    if self.db_manager.insert_evidence(file_path, file_size, last_modified, sha256_hash, md5_hash, shannon_entropy):
                        print(f"  Metadata recorded for {file_path}")
                        log_entry = self.custody_logger.log_event(f"Metadata recorded for {file_path}")
                        self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

                        # Add file to internal list for later archiving
                        if self.archive_manager.add_to_archive(file_path):
                            print(f"  Marked for archiving: {file_path}")
                            log_entry = self.custody_logger.log_event(f"File marked for archiving: {file_path}")
                            self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
                    log_entry = self.custody_logger.log_event(f"Error processing file {file_path}: {e}")
                    self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

        end_timestamp = datetime.now()
        duration = end_timestamp - self.start_timestamp

        # Create the final archive after all files have been processed
        print("Finalizing archive...")
        if self.archive_manager.create_final_archive():
            log_entry = self.custody_logger.log_event(f"Final evidence archive created at: {self.config.ARCHIVE_PATH}")
            self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
        else:
            log_entry = self.custody_logger.log_event(f"Failed to create final evidence archive.")
            self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

        # Log agent completion event
        log_entry = self.custody_logger.log_event(f"Forensic agent scan completed in {duration}. Archive: {self.config.ARCHIVE_PATH}, Database: {self.config.DB_PATH}")
        self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
        print("Forensic scan completed.")

    def get_summary(self):
        print("--- Forensic Scan Summary ---")
        evidence_count = len(self.db_manager.get_all_evidence())
        log_count = len(self.db_manager.get_all_custody_logs())
        print(f"Total evidence files recorded: {evidence_count}")
        print(f"Total custody log entries: {log_count}")
        print(f"Evidence database: {self.config.DB_PATH}")
        print(f"Evidence archive: {self.config.ARCHIVE_PATH}")
        print("-----------------------------")
```

## Explanation of Individual Components:

### 2.1. CONFIGURATION (`Config` Class)
This class centralizes all the essential settings for the forensic agent, such as the root directory to scan, paths for the database and archive, the secret key for HMAC signing, target file extensions, and a MIME type map for file identification. It acts as the control panel for the agent's behavior.

### 2.2. ENTROPY (SHANNON) (`EntropyCalculator` Class)
This class calculates the Shannon entropy of files. High entropy values often indicate encrypted or compressed data, which is a significant clue in forensic analysis. It reads a sample of the file to estimate entropy in bits per byte.

### 2.3. DIRECTORY SCANNER (BFS + GENERATORS) (`DirectoryScanner` Class)
Implements a Breadth-First Search (BFS) algorithm to traverse the file system. It uses Python generators, which make it memory-efficient by yielding file paths one by one rather than loading all paths into memory at once.

### 2.4. FILE IDENTIFIER (EXTENSION + MAGIC NUMBER) (`FileIdentifier` Class)
Determines if a file is relevant for forensic analysis using a two-stage approach:
1.  **Extension Filter**: Checks if the file's extension matches a predefined list.
2.  **Magic Number / MIME Verification**: If `python-magic` is available, it verifies the file's true type based on its content (magic numbers), comparing it against expected MIME types to prevent disguised files.

### 2.5. HASHER (SHA-256 + MD5) (`Hasher` Class)
Computes cryptographic hash values (SHA-256 and MD5) for files. Hashes are crucial for verifying file integrity and quickly identifying known files or modifications.

### 2.6. SQLITE METADATA STORE (`DatabaseManager` Class)
Manages all interactions with the SQLite database (`evidence.db`). It's responsible for:
*   Initializing `evidence` (file metadata) and `custody_log` tables.
*   Inserting file metadata (path, size, hashes, entropy, etc.).
*   Logging custody events.
SQLite is chosen for its lightweight, server-less nature, suitable for embedded applications.

### 2.7. ARCHIVE MANAGER (TAR/GZ) (`ArchiveManager` Class)
Handles the creation of a compressed archive (`.tar.gz`) for all collected evidence files. This is essential for preserving evidence, maintaining integrity, and facilitating transportability. Files are collected during the scan and archived in a single operation at the end to ensure consistency.

### 2.8. CUSTODY LOGGER (HMAC) (`CustodyLogger` Class)
Maintains the integrity and trustworthiness of the forensic process by logging every significant action. Log entries are cryptographically signed using HMAC (Hash-based Message Authentication Code) to guarantee authenticity and prevent tampering.

### 2.9. ACCESS CONTROL (`AccessControl` Class)
Provides graceful handling of file system permissions. It checks for read access and logs permission errors, allowing the agent to continue processing other accessible data without crashing.

### 2.10. FORENSIC AGENT ORCHESTRATION (`ForensicAgent` Class)
This is the core class that orchestrates the entire forensic workflow. It integrates all the above components to:
1.  Initialize all modules based on `Config`.
2.  Scan the file system.
3.  Identify relevant files.
4.  Analyze files (hashing, entropy).
5.  Store metadata in SQLite.
6.  Archive physical evidence.
7.  Log all events with HMAC signatures.
8.  Handle errors, especially permission issues.

## Execution and Verification

### Clean-up and Re-running the Agent
Before the final run, any existing `evidence.db` and `evidence_archive.tar.gz` files were removed to ensure a fresh scan.

```python
import os
from pathlib import Path

# Clean up previous artifacts
if Path('evidence.db').exists():
    os.remove('evidence.db')
    print("Removed existing evidence.db")

if Path('evidence_archive.tar.gz').exists():
    os.remove('evidence_archive.tar.gz')
    print("Removed existing evidence_archive.tar.gz")

# Instantiate the Config object
config = Config()

# Instantiate the ForensicAgent
forensic_agent = ForensicAgent(config)

# Run the scan
forensic_agent.run_scan()

# Get a summary of the scan results
forensic_agent.get_summary()
```

**Output from running the Forensic Agent:**
```
Removed existing evidence.db
Removed existing evidence_archive.tar.gz
Forensic Agent initialised. Scanning from: .
Starting forensic scan...
Found relevant file: Executable codes Assig 2.txt
  Metadata recorded for Executable codes Assig 2.txt
  Marked for archiving: Executable codes Assig 2.txt
Finalizing archive...
Final archive created at: /content/evidence_archive.tar.gz
Forensic scan completed.
--- Forensic Scan Summary ---
Total evidence files recorded: 1
Total custody log entries: 5
Evidence database: evidence.db
Evidence archive: evidence_archive.tar.gz
-----------------------------
```

### Inspecting the `evidence.db` Database

```python
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect('evidence.db')

# Query the evidence table
evidence_df = pd.read_sql_query("SELECT * FROM evidence", conn)

print("--- Evidence Table Contents ---")
display(evidence_df)

# Query the custody_log table
custody_df = pd.read_sql_query("SELECT * FROM custody_log", conn)

print("\n--- Custody Log Table Contents ---")
display(custody_df)

# Close the connection
conn.close()
```

**Output: Evidence Table Contents**

| id | file_path                    | file_size | last_modified              | sha256_hash                                                        | md5_hash                         | shannon_entropy | timestamp                  |
|----|------------------------------|-----------|----------------------------|--------------------------------------------------------------------|----------------------------------|-----------------|----------------------------|
| 1  | Executable codes Assig 2.txt | 11467     | 2026-04-08T15:39:32.716436 | fa3ff18f8799560a3425c07271d07a4ac8faf16c70ce48... | cbfbdfe9110bce82fc7477fd32094fb4 | 4.760033        | 2026-04-08T19:42:18.479766 |

**Output: Custody Log Table Contents**

| id | timestamp                  | event_description                                                          | hmac_signature                                                   |
|----|----------------------------|----------------------------------------------------------------------------|------------------------------------------------------------------|
| 1  | 2026-04-08T19:42:18.469374 | Forensic agent initiated scan.                                             | 295023df5a5303f285d65044ef4a13918b54f510f8ff90... |
| 2  | 2026-04-08T19:42:18.488894 | Metadata recorded for Executable codes Assig 2.txt                         | 72a94b8017b8d1adb6edb904fb09c27b567eaa77dd0d78... |
| 3  | 2026-04-08T19:42:18.496969 | File marked for archiving: Executable codes Assig 2.txt                  | da7d0583c9276a87699ed9659e418a58945c756df3fe49... |
| 4  | 2026-04-08T19:42:18.507219 | Final evidence archive created at: evidence_archive.tar.gz                 | 5c25c3328e1d166715c73268c37287ebe855af5a8631dd... |
| 5  | 2026-04-08T19:42:18.516539 | Forensic agent scan completed in 0:00:00.03639...                          | 156a8669061776ed372df2683eadb43171e6b0e70fccd4... |

### Inspecting the `evidence_archive.tar.gz` Archive

```python
import tarfile
from pathlib import Path

archive_path = Path('evidence_archive.tar.gz')

if archive_path.exists():
    print(f"Contents of '{archive_path}':")
    try:
        with tarfile.open(archive_path, 'r:gz') as tar:
            for member in tar.getmembers():
                print(f"- {member.name}")
    except Exception as e:
        print(f"Error reading archive: {e}")
else:
    print(f"Archive '{archive_path}' does not exist.")
```

**Output: Contents of `evidence_archive.tar.gz`**
```
Contents of 'evidence_archive.tar.gz':
- Executable codes Assig 2.txt
```

## Conclusion
The digital forensics agent has been successfully implemented, tested, and verified according to the assignment requirements. It effectively scans for relevant files, collects crucial metadata (hashes, entropy), maintains a secure chain of custody through HMAC-signed logs, and archives identified evidence. The generated database (`evidence.db`) and archive (`evidence_archive.tar.gz`) provide concrete proof of the agent's functionality and can be directly used for your assignment.

In [43]:
import os
from pathlib import Path

# Clean up previous artifacts
if Path('evidence.db').exists():
    os.remove('evidence.db')
    print("Removed existing evidence.db")

if Path('evidence_archive.tar.gz').exists():
    os.remove('evidence_archive.tar.gz')
    print("Removed existing evidence_archive.tar.gz")

Removed existing evidence.db
Removed existing evidence_archive.tar.gz


In [44]:
import os
from pathlib import Path

# Instantiate the Config object (defined in the consolidated code block)
config = Config()

# Instantiate the ForensicAgent
forensic_agent = ForensicAgent(config)

# Run the scan
forensic_agent.run_scan()

# Get a summary of the scan results
forensic_agent.get_summary()

Forensic Agent initialised. Scanning from: .
Starting forensic scan...
Found relevant file: Executable codes Assig 2.txt
  Metadata recorded for Executable codes Assig 2.txt
  Marked for archiving: Executable codes Assig 2.txt
Finalizing archive...
Final archive created at: /content/evidence_archive.tar.gz
Forensic scan completed.
--- Forensic Scan Summary ---
Total evidence files recorded: 1
Total custody log entries: 5
Evidence database: evidence.db
Evidence archive: evidence_archive.tar.gz
-----------------------------


In [45]:
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect('evidence.db')

# Query the evidence table
evidence_df = pd.read_sql_query("SELECT * FROM evidence", conn)

print("--- Evidence Table Contents ---")
display(evidence_df)

# Query the custody_log table
custody_df = pd.read_sql_query("SELECT * FROM custody_log", conn)

print("\n--- Custody Log Table Contents ---")
display(custody_df)

# Close the connection
conn.close()

--- Evidence Table Contents ---


,id,file_path,file_size,last_modified,sha256_hash,md5_hash,shannon_entropy,timestamp
0,1,Executable codes Assig 2.txt,11467,2026-04-08T15:39:32.716436,fa3ff18f8799560a3425c07271d07a4ac8faf16c70ce48...,cbfbdfe9110bce82fc7477fd32094fb4,4.760033,2026-04-08T19:42:18.479766



--- Custody Log Table Contents ---


,id,timestamp,event_description,hmac_signature
0,1,2026-04-08T19:42:18.469374,Forensic agent initiated scan.,295023df5a5303f285d65044ef4a13918b54f510f8ff90...
1,2,2026-04-08T19:42:18.488894,Metadata recorded for Executable codes Assig 2...,72a94b8017b8d1adb6edb904fb09c27b567eaa77dd0d78...
2,3,2026-04-08T19:42:18.496969,File marked for archiving: Executable codes As...,da7d0583c9276a87699ed9659e418a58945c756df3fe49...
3,4,2026-04-08T19:42:18.507219,Final evidence archive created at: evidence_ar...,5c25c3328e1d166715c73268c37287ebe855af5a8631dd...
4,5,2026-04-08T19:42:18.516539,Forensic agent scan completed in 0:00:00.03639...,156a8669061776ed372df2683eadb43171e6b0e70fccd4...


In [46]:
import tarfile
from pathlib import Path

archive_path = Path('evidence_archive.tar.gz')

if archive_path.exists():
    print(f"Contents of '{archive_path}':")
    try:
        with tarfile.open(archive_path, 'r:gz') as tar:
            for member in tar.getmembers():
                print(f"- {member.name}")
    except Exception as e:
        print(f"Error reading archive: {e}")
else:
    print(f"Archive '{archive_path}' does not exist.")

Contents of 'evidence_archive.tar.gz':
- Executable codes Assig 2.txt


In [39]:
import os
from pathlib import Path

# Clean up previous artifacts
if Path('evidence.db').exists():
    os.remove('evidence.db')
    print("Removed existing evidence.db")

if Path('evidence_archive.tar.gz').exists():
    os.remove('evidence_archive.tar.gz')
    print("Removed existing evidence_archive.tar.gz")


Removed existing evidence.db


In [40]:
import os
from pathlib import Path

# Instantiate the Config object (defined in the consolidated code block)
config = Config()

# Instantiate the ForensicAgent
forensic_agent = ForensicAgent(config)

# Run the scan
forensic_agent.run_scan()

# Get a summary of the scan results
forensic_agent.get_summary()

Forensic Agent initialised. Scanning from: .
Starting forensic scan...
Found relevant file: Executable codes Assig 2.txt
  Metadata recorded for Executable codes Assig 2.txt
  Marked for archiving: Executable codes Assig 2.txt
Finalizing archive...
Final archive created at: /content/evidence_archive.tar.gz
Forensic scan completed.
--- Forensic Scan Summary ---
Total evidence files recorded: 1
Total custody log entries: 5
Evidence database: evidence.db
Evidence archive: evidence_archive.tar.gz
-----------------------------


In [41]:
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect('evidence.db')

# Query the evidence table
evidence_df = pd.read_sql_query("SELECT * FROM evidence", conn)

print("--- Evidence Table Contents ---")
display(evidence_df)

# Query the custody_log table
custody_df = pd.read_sql_query("SELECT * FROM custody_log", conn)

print("\n--- Custody Log Table Contents ---")
display(custody_df)

# Close the connection
conn.close()

--- Evidence Table Contents ---


,id,file_path,file_size,last_modified,sha256_hash,md5_hash,shannon_entropy,timestamp
0,1,Executable codes Assig 2.txt,11467,2026-04-08T15:39:32.716436,fa3ff18f8799560a3425c07271d07a4ac8faf16c70ce48...,cbfbdfe9110bce82fc7477fd32094fb4,4.760033,2026-04-08T19:27:45.629253



--- Custody Log Table Contents ---


,id,timestamp,event_description,hmac_signature
0,1,2026-04-08T19:27:45.618749,Forensic agent initiated scan.,164cfaf6066b70ddfef58cb530ab66cecc39cdc368e7fc...
1,2,2026-04-08T19:27:45.638499,Metadata recorded for Executable codes Assig 2...,111155ed0d7309dfe2c913e24f6c828d6abcecc1d7adc1...
2,3,2026-04-08T19:27:45.646479,File marked for archiving: Executable codes As...,aa79ff9618aad2c8285b4a2fc10d64a86ab62a4d9dd3d5...
3,4,2026-04-08T19:27:45.657763,Final evidence archive created at: evidence_ar...,f79f0068cb635d369ad27479329d58d0d3c10db2e3b875...
4,5,2026-04-08T19:27:45.668815,Forensic agent scan completed in 0:00:00.03750...,4772b44b3782f91be83411f864b7ff9bf29ad92a36ff7f...


In [42]:
import tarfile
from pathlib import Path

archive_path = Path('evidence_archive.tar.gz')

if archive_path.exists():
    print(f"Contents of '{archive_path}':")
    try:
        with tarfile.open(archive_path, 'r:gz') as tar:
            for member in tar.getmembers():
                print(f"- {member.name}")
    except Exception as e:
        print(f"Error reading archive: {e}")
else:
    print(f"Archive '{archive_path}' does not exist.")

Contents of 'evidence_archive.tar.gz':
- Executable codes Assig 2.txt


In [38]:
import tarfile
from pathlib import Path

archive_path = Path('evidence_archive.tar.gz')

if archive_path.exists():
    print(f"Contents of '{archive_path}':")
    try:
        with tarfile.open(archive_path, 'r:gz') as tar:
            for member in tar.getmembers():
                print(f"- {member.name}")
    except Exception as e:
        print(f"Error reading archive: {e}")
else:
    print(f"Archive '{archive_path}' does not exist.")

Archive 'evidence_archive.tar.gz' does not exist.


In [36]:
import os
from pathlib import Path

# Instantiate the Config object (defined in the consolidated code block)
config = Config()

# Instantiate the ForensicAgent
forensic_agent = ForensicAgent(config)

# Run the scan
forensic_agent.run_scan()

# Get a summary of the scan results
forensic_agent.get_summary()

Forensic Agent initialised. Scanning from: .
Starting forensic scan...
Found relevant file: Executable codes Assig 2.txt
Finalizing archive...
No files to archive.
Forensic scan completed.
--- Forensic Scan Summary ---
Total evidence files recorded: 1
Total custody log entries: 9
Evidence database: evidence.db
Evidence archive: evidence_archive.tar.gz
-----------------------------


### 2.2. ENTROPY (SHANNON)

This section introduces the `EntropyCalculator` class, which provides functionality to estimate the Shannon entropy of a file. Shannon entropy is a measure of the unpredictability or randomness of data. In digital forensics, a high entropy value can sometimes indicate that a file is either compressed or encrypted, as these processes tend to maximise randomness.

In [7]:
import math
from pathlib import Path

class EntropyCalculator:
    def __init__(self, sample_size: int = 1024 * 64):
        # sample_size limits how much of the file we read for entropy estimation
        self.sample_size = sample_size

    def file_entropy(self, path: Path) -> float:
        """
        Estimate Shannon entropy (in bits per byte) for the given file.
        High values may indicate encryption or compression.
        """
        try:
            with path.open("rb") as f:
                data = f.read(self.sample_size)
        except Exception:
            return 0.0

        if not data:
            return 0.0

        freq = [0] * 256
        for b in data:
            freq[b] += 1

        entropy = 0.0
        length = len(data)

        for count in freq:
            if count == 0:
                continue
            p = count / length
            entropy -= p * math.log2(p)

        return entropy

### Explanation of `EntropyCalculator` Class:

*   **`import math`**: Necessary for mathematical operations, specifically `math.log2` for calculating logarithms base 2.
*   **`from pathlib import Path`**: Imported to work with file paths in an object-oriented manner, similar to its use in the `FileIdentifier`.
*   **`__init__(self, sample_size: int = 1024 * 64)`**: The constructor initializes the `sample_size`, which is the maximum number of bytes to read from a file to estimate its entropy. By default, it's set to 64 KB, balancing accuracy with performance for large files.
*   **`file_entropy(self, path: Path) -> float`**: This method calculates the Shannon entropy for a given file:
    1.  **File Reading**: It attempts to open the file in binary read mode (`"rb"`) and reads up to `self.sample_size` bytes. A `try-except` block handles potential errors during file access, returning `0.0` if the file can't be read.
    2.  **Empty File Check**: If no data is read (e.g., an empty file), it returns `0.0`.
    3.  **Frequency Count**: It creates a frequency array (`freq`) for all 256 possible byte values (0-255). It then iterates through the read `data` and increments the count for each byte value encountered.
    4.  **Entropy Calculation**: It iterates through the `freq` array:
        *   For each byte count, it calculates the probability `p` of that byte appearing (`count / length`).
        *   It then applies the Shannon entropy formula: `entropy -= p * math.log2(p)`. The sum of `p * log2(p)` for all byte probabilities gives the entropy.
    5.  **Return Value**: The method returns the calculated entropy as a floating-point number, representing bits per byte. A value close to 8 (for an 8-bit byte) indicates high randomness (like encrypted or highly compressed data), while lower values suggest structured or repetitive data.

### 2.3. DIRECTORY SCANNER (BFS + GENERATORS)

This section implements the `DirectoryScanner` class, responsible for traversing the file system. It uses a Breadth-First Search (BFS) approach, which explores all nodes at the present depth level before moving on to nodes at the next depth level. The use of Python generators makes this process memory-efficient, especially for large directory structures, as it yields files one by one instead of loading all file paths into memory at once.

In [8]:
from collections import deque
from pathlib import Path

class DirectoryScanner:
    def __init__(self, root: Path):
        self.root = Path(root)

    def bfs_scan(self):
        """
        Breadth-first traversal using a generator to keep memory usage stable.
        """
        queue = deque([self.root])
        while queue:
            current = queue.popleft()
            if not current.exists():
                continue
            for entry in current.iterdir():
                if entry.is_dir():
                    queue.append(entry)
                else:
                    yield entry

### Explanation of `DirectoryScanner` Class:

*   **`from collections import deque`**: Imports `deque` (double-ended queue) from the `collections` module. `deque` is preferred over a standard list for queue operations (appending and popping from opposite ends) because it provides `O(1)` (constant time) performance for these operations, which is efficient for BFS.
*   **`from pathlib import Path`**: Imports the `Path` object for working with file system paths in an object-oriented way.
*   **`__init__(self, root: Path)`**: The constructor initializes the scanner with a `root` directory, converting it to a `Path` object to ensure consistent handling.
*   **`bfs_scan(self)`**: This is the core method that performs the Breadth-First Search:
    1.  **Initialize Queue**: A `deque` named `queue` is initialized with the `self.root` directory, which is the starting point of the scan.
    2.  **BFS Loop**: The `while queue:` loop continues as long as there are directories to visit.
    3.  **Current Directory**: `current = queue.popleft()` retrieves the oldest directory from the front of the queue to process it.
    4.  **Existence Check**: `if not current.exists(): continue` ensures that the directory still exists before attempting to iterate over its contents. This handles cases where a directory might have been deleted during the scan.
    5.  **Iterate Contents**: `for entry in current.iterdir():` iterates through all entries (files and subdirectories) within the `current` directory.
    6.  **Directory Handling**: `if entry.is_dir(): queue.append(entry)`: If an entry is a directory, it's added to the back of the queue to be processed later, ensuring BFS order.
    7.  **File Handling**: `else: yield entry`: If an entry is a file, it is `yield`ed. This is the generator aspect; instead of building a large list of all files, `bfs_scan` provides files one by one as they are found, making it very memory efficient. The calling code can then iterate over the results of `bfs_scan`.

### 2.6. SQLITE METADATA STORE

This section implements the `DatabaseManager` class, which handles all interactions with the SQLite database. SQLite is an excellent choice for embedded, lightweight databases like this, as it doesn't require a separate server process and stores the entire database in a single file. The `DatabaseManager` ensures that:
*   The database and tables are properly initialized.
*   Metadata about identified files (path, hashes, entropy, etc.) is stored.
*   Audit log entries (custody events) are recorded.

Using a database centralizes the collected forensic information, making it easy to query and manage.

In [10]:
import sqlite3
from pathlib import Path
from datetime import datetime

class DatabaseManager:
    def __init__(self, db_path: Path):
        self.db_path = db_path
        self._create_tables()

    def _create_tables(self):
        """
        Initialises the SQLite database with necessary tables.
        """
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()

        # Evidence table for file metadata
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS evidence (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                file_path TEXT NOT NULL UNIQUE,
                file_size INTEGER,
                last_modified TEXT,
                sha256_hash TEXT,
                md5_hash TEXT,
                shannon_entropy REAL,
                timestamp TEXT NOT NULL
            )
        """)

        # Custody log table for audit trail
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS custody_log (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT NOT NULL,
                event_description TEXT NOT NULL,
                hmac_signature TEXT NOT NULL
            )
        """)
        conn.commit()
        conn.close()

    def insert_evidence(self, file_path: Path, file_size: int, last_modified: datetime, sha256_hash: str, md5_hash: str, shannon_entropy: float):
        """
        Inserts file metadata into the evidence table.
        """
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        timestamp = datetime.now().isoformat()
        try:
            cursor.execute("""
                INSERT INTO evidence (file_path, file_size, last_modified, sha256_hash, md5_hash, shannon_entropy, timestamp)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (str(file_path), file_size, last_modified.isoformat(), sha256_hash, md5_hash, shannon_entropy, timestamp))
            conn.commit()
            return True
        except sqlite3.IntegrityError: # Handle UNIQUE constraint violation for file_path
            print(f"Warning: Evidence for {file_path} already exists. Skipping insertion.")
            return False
        except Exception as e:
            print(f"Error inserting evidence for {file_path}: {e}")
            return False
        finally:
            conn.close()

    def log_custody_event(self, event_description: str, hmac_signature: str):
        """
        Logs a custody event with an HMAC signature into the custody_log table.
        """
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        timestamp = datetime.now().isoformat()
        try:
            cursor.execute("""
                INSERT INTO custody_log (timestamp, event_description, hmac_signature)
                VALUES (?, ?, ?)
            """, (timestamp, event_description, hmac_signature))
            conn.commit()
            return True
        except Exception as e:
            print(f"Error logging custody event: {e}")
            return False
        finally:
            conn.close()

    def get_all_evidence(self) -> list:
        """
        Retrieves all evidence records.
        """
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM evidence")
        records = cursor.fetchall()
        conn.close()
        return records

    def get_all_custody_logs(self) -> list:
        """
        Retrieves all custody log records.
        """
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM custody_log")
        records = cursor.fetchall()
        conn.close()
        return records


### Explanation of `DatabaseManager` Class:

*   **`import sqlite3`**: Imports Python's standard library module for SQLite database interaction.
*   **`from pathlib import Path`**: For path handling.
*   **`from datetime import datetime`**: For recording timestamps.
*   **`__init__(self, db_path: Path)`**: The constructor takes the path to the SQLite database file and immediately calls `_create_tables()` to ensure the database structure is set up.
*   **`_create_tables(self)`**: This private helper method creates two tables if they don't already exist:
    *   **`evidence`**: Stores metadata about each identified file, including its unique path, size, last modified date, SHA-256 and MD5 hashes, Shannon entropy, and the timestamp of when it was recorded.
    *   **`custody_log`**: Records audit events, each with a timestamp, a description of the event, and an HMAC signature to ensure its integrity (discussed further in the custody section).
*   **`insert_evidence(...)`**: This method takes various pieces of file metadata and inserts them into the `evidence` table. It includes error handling for `sqlite3.IntegrityError` (which would occur if trying to insert a file with a `file_path` that already exists, as `file_path` is `UNIQUE`) and general exceptions.
*   **`log_custody_event(self, event_description: str, hmac_signature: str)`**: This method inserts a new entry into the `custody_log` table, recording the event description and its HMAC signature.
*   **`get_all_evidence(self)`** and **`get_all_custody_logs(self)`**: These utility methods are provided to retrieve all records from their respective tables, which can be useful for reporting or further analysis. Each method establishes a new connection to the database, executes the query, fetches all results, and then closes the connection.

### 2.7. ARCHIVE MANAGER (TAR/GZ)

This section introduces the `ArchiveManager` class, which is responsible for creating a compressed archive of all the collected evidence files. Archiving is a critical step in digital forensics to:
*   **Preserve Evidence**: Collect all relevant files into a single, immutable package.
*   **Maintain Integrity**: Often, archives are hashed themselves to ensure their integrity.
*   **Transportability**: Easier to transfer and store a single archive than many individual files.
*   **Space Efficiency**: Compression (`.tar.gz`) reduces the storage footprint.

In [11]:
import tarfile
from pathlib import Path

class ArchiveManager:
    def __init__(self, archive_path: Path):
        self.archive_path = archive_path

    def add_to_archive(self, file_path: Path):
        """
        Adds a specified file to the TAR.GZ archive.
        If the archive does not exist, it will be created.
        """
        try:
            # Use 'a:gz' mode for append (if exists) or create (if not exists) with gzip compression
            with tarfile.open(self.archive_path, 'a:gz') as tar:
                # Add the file, preserving its path relative to the current directory
                # arcname ensures the path inside the archive is relative and clean
                tar.add(file_path, arcname=file_path.relative_to(Path.cwd()))
            return True
        except Exception as e:
            print(f"Error adding {file_path} to archive: {e}")
            return False

    def extract_archive(self, extract_path: Path):
        """
        Extracts the entire archive to a specified path.
        """
        try:
            with tarfile.open(self.archive_path, 'r:gz') as tar:
                tar.extractall(path=extract_path)
            print(f"Archive extracted to {extract_path}")
            return True
        except Exception as e:
            print(f"Error extracting archive: {e}")
            return False

### Explanation of `ArchiveManager` Class:

*   **`import tarfile`**: Imports Python's `tarfile` module, which provides functionality to read and write TAR archives, including those compressed with gzip (`.tar.gz`).
*   **`from pathlib import Path`**: For consistent path handling.
*   **`__init__(self, archive_path: Path)`**: The constructor takes the `archive_path` where the evidence archive will be created or managed.
*   **`add_to_archive(self, file_path: Path)`**: This method adds a single file to the `.tar.gz` archive:
    *   It opens the `tarfile` in `'a:gz'` mode: `a` stands for append (if the archive exists, files are added to it) or create (if it doesn't exist), and `gz` specifies gzip compression.
    *   `tar.add(file_path, arcname=file_path.relative_to(Path.cwd()))` adds the file. The `arcname` argument is crucial here: it stores the file in the archive with a path relative to the current working directory, making the archive more portable and preventing absolute paths from being embedded. This assumes the `file_path` is relative to the `Path.cwd()` (current working directory).
    *   Includes a `try-except` block for error handling during the archiving process.
*   **`extract_archive(self, extract_path: Path)`**: This utility method extracts all contents of the archive to a specified `extract_path`.
    *   It opens the `tarfile` in `'r:gz'` mode (read with gzip compression).
    *   `tar.extractall(path=extract_path)` extracts all members. The `path` argument specifies the destination directory for extraction.

### 2.8. CUSTODY LOGGER (HMAC)

This section implements the `CustodyLogger` class, which is vital for maintaining the integrity and trustworthiness of the forensic process. A chain of custody log records every significant action taken with the evidence. To prevent tampering and ensure authenticity, these log entries are cryptographically signed using HMAC (Hash-based Message Authentication Code).

**HMAC Benefits:**
*   **Integrity**: Guarantees that the log entry has not been altered since it was signed.
*   **Authenticity**: Verifies that the log entry was created by someone possessing the secret key.

This adds a layer of cryptographic assurance to the audit trail, which is critical in legal and investigative contexts.

In [12]:
import hmac
import hashlib
from datetime import datetime

class CustodyLogger:
    def __init__(self, secret_key: bytes):
        self.secret_key = secret_key

    def create_hmac_signature(self, data: str) -> str:
        """
        Generates an HMAC-SHA256 signature for the given data.
        """
        h = hmac.new(self.secret_key, data.encode('utf-8'), hashlib.sha256)
        return h.hexdigest()

    def verify_hmac_signature(self, data: str, signature: str) -> bool:
        """
        Verifies an HMAC-SHA256 signature against the given data.
        """
        expected_signature = self.create_hmac_signature(data)
        return hmac.compare_digest(expected_signature, signature)

    def log_event(self, event_description: str) -> dict:
        """
        Creates a signed custody log entry.
        """
        timestamp = datetime.now().isoformat()
        log_data = f"{timestamp}|{event_description}"
        signature = self.create_hmac_signature(log_data)
        return {
            "timestamp": timestamp,
            "event_description": event_description,
            "hmac_signature": signature,
            "log_data_for_verification": log_data # For easier external verification if needed
        }

### Explanation of `CustodyLogger` Class:

*   **`import hmac`** and **`import hashlib`**: These modules are essential for creating and verifying HMACs. `hmac` provides the HMAC algorithm implementation, and `hashlib` provides the underlying cryptographic hash function (SHA-256 in this case).
*   **`from datetime import datetime`**: Used for generating timestamps for log entries.
*   **`__init__(self, secret_key: bytes)`**: The constructor takes a `secret_key` (as bytes) which is crucial for HMAC operations. This key must be kept confidential and is used for both signing and verifying log entries. It corresponds to the `CUSTODY_KEY` in the `Config`.
*   **`create_hmac_signature(self, data: str) -> str`**: This method generates the HMAC signature:
    *   It takes a string `data` (which will be the timestamped event description).
    *   `hmac.new()` creates a new HMAC object, configured with the `secret_key`, the data (encoded to bytes), and the SHA-256 hash algorithm.
    *   `h.hexdigest()` returns the hexadecimal representation of the computed HMAC.
*   **`verify_hmac_signature(self, data: str, signature: str) -> bool`**: This method verifies if a given `signature` matches the `data`:
    *   It re-generates the expected signature using `create_hmac_signature` with the provided `data`.
    *   `hmac.compare_digest()` is used to safely compare the two signatures. This function is designed to prevent timing attacks, which could reveal information about the secret key if a regular string comparison (`==`) were used.
*   **`log_event(self, event_description: str) -> dict`**: This method creates a complete, signed custody log entry:
    *   It gets the current `timestamp` in ISO format.
    *   It combines the `timestamp` and `event_description` into a `log_data` string, which will be the content signed by HMAC.
    *   It generates an `hmac_signature` for this `log_data`.
    *   Finally, it returns a dictionary containing the timestamp, event description, the generated HMAC signature, and the raw `log_data` (for easy external verification if needed). This dictionary would then typically be passed to the `DatabaseManager` to be stored in the `custody_log` table.

### 2.9. ACCESS CONTROL

This section implements the `AccessControl` class, which is designed to handle file system permissions gracefully. In digital forensics, agents often encounter files or directories with restricted access. Instead of crashing, a robust agent should identify these issues, log them, and continue processing other accessible data. This class provides a mechanism to check and manage such access rights.

In [13]:
import os
from pathlib import Path

class AccessControl:
    def __init__(self):
        pass # Currently no state to initialize

    def has_read_access(self, path: Path) -> bool:
        """
        Checks if the agent has read access to a given path.
        """
        try:
            return os.access(path, os.R_OK)
        except Exception as e:
            print(f"Warning: Could not check read access for {path}: {e}")
            return False

    def handle_permission_error(self, path: Path, action: str):
        """
        Logs a permission error for a given path and action.
        This could be expanded to integrate with a CustodyLogger.
        """
        print(f"Permission Error: Cannot {action} {path}. Skipping.")
        # In a full agent, this would log to custody_log or a dedicated error log

### Explanation of `AccessControl` Class:

*   **`import os`**: Imports the `os` module, which provides a way of using operating system dependent functionality, including file system access checks.
*   **`from pathlib import Path`**: For consistent path handling.
*   **`__init__(self)`**: The constructor currently does not require any specific initialization, as its methods primarily rely on built-in `os` functions.
*   **`has_read_access(self, path: Path) -> bool`**: This method checks if the current process has read permissions for a given file or directory:
    *   `os.access(path, os.R_OK)` is the core function call. `os.R_OK` is a constant representing the read access mode.
    *   It returns `True` if read access is granted, `False` otherwise. A `try-except` block is included to catch any unexpected errors during the access check itself, returning `False` if an error occurs.
*   **`handle_permission_error(self, path: Path, action: str)`**: This method is a placeholder for handling permission errors:
    *   It prints a warning message indicating that an `action` (e.g., 'read', 'scan') could not be performed on the given `path` due to permissions.
    *   The comment `In a full agent, this would log to custody_log or a dedicated error log` highlights that in a complete forensic agent, this method would integrate with the `CustodyLogger` (or a separate error logging mechanism) to record these events formally. This ensures that any skipped files due to permissions are documented in the audit trail.

### 2.10. FORENSIC AGENT ORCHESTRATION

This is the core `ForensicAgent` class, bringing together all the previously defined modules (`Config`, `DirectoryScanner`, `FileIdentifier`, `Hasher`, `EntropyCalculator`, `DatabaseManager`, `ArchiveManager`, `CustodyLogger`, `AccessControl`) into a cohesive workflow. This class defines the overall process of scanning, identifying, analyzing, and preserving evidence.

The orchestration involves:
1.  **Initialization**: Setting up all component instances.
2.  **Scanning**: Traversing the specified root directory.
3.  **Identification**: Filtering relevant files.
4.  **Analysis**: Computing hashes and entropy.
5.  **Storage**: Saving metadata to SQLite and archiving physical files.
6.  **Logging**: Maintaining an HMAC-signed chain of custody.
7.  **Error Handling**: Gracefully managing permission issues.

This `agent.py` equivalent acts as the brain of our digital forensics solution.

In [17]:
import os
from pathlib import Path
from datetime import datetime
import hashlib

# Assuming all previous classes (Config, EntropyCalculator, DirectoryScanner,
# FileIdentifier, Hasher, DatabaseManager, ArchiveManager, CustodyLogger, AccessControl)
# are already defined and available in the execution environment.
# For the Colab single-file version, they would typically precede this class definition.

# ==========================
# HASHER (SHA-256 + MD5)
# ==========================
class Hasher:
    def hash_file(self, path: Path, buffer_size: int = 65536) -> dict:
        """
        Computes SHA-256 and MD5 hashes for a given file.
        """
        sha256_hash = hashlib.sha256()
        md5_hash = hashlib.md5()
        try:
            with open(path, "rb") as f:
                while True:
                    data = f.read(buffer_size)
                    if not data:
                        break
                    sha256_hash.update(data)
                    md5_hash.update(data)
            return {
                "sha256": sha256_hash.hexdigest(),
                "md5": md5_hash.hexdigest()
            }
        except Exception:
            return {"sha256": "ERROR", "md5": "ERROR"}

class ForensicAgent:
    def __init__(self, config: 'Config'):
        self.config = config
        self.scanner = DirectoryScanner(config.ROOT)
        self.identifier = FileIdentifier(config.TARGET_EXTENSIONS, config.MIME_MAP)
        self.hasher = Hasher()
        self.entropy_calculator = EntropyCalculator()
        self.db_manager = DatabaseManager(config.DB_PATH)
        self.archive_manager = ArchiveManager(config.ARCHIVE_PATH)
        self.custody_logger = CustodyLogger(config.CUSTODY_KEY)
        self.access_control = AccessControl()
        self.start_timestamp = datetime.now()

        # Log agent start event
        log_entry = self.custody_logger.log_event("Forensic agent initiated scan.")
        self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
        print(f"Forensic Agent initialised. Scanning from: {self.config.ROOT}")

    def run_scan(self):
        print("Starting forensic scan...")
        for file_path in self.scanner.bfs_scan():
            if not self.access_control.has_read_access(file_path):
                self.access_control.handle_permission_error(file_path, "read")
                log_entry = self.custody_logger.log_event(f"Permission denied for file: {file_path}. Skipped.")
                self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
                continue

            if self.identifier.is_relevant(file_path):
                print(f"Found relevant file: {file_path}")
                # Get file metadata
                try:
                    file_stat = file_path.stat()
                    file_size = file_stat.st_size
                    last_modified = datetime.fromtimestamp(file_stat.st_mtime)

                    # Compute hashes
                    hashes = self.hasher.hash_file(file_path)
                    sha256_hash = hashes['sha256']
                    md5_hash = hashes['md5']

                    # Compute entropy
                    shannon_entropy = self.entropy_calculator.file_entropy(file_path)

                    # Insert metadata into DB
                    if self.db_manager.insert_evidence(file_path, file_size, last_modified, sha256_hash, md5_hash, shannon_entropy):
                        print(f"  Metadata recorded for {file_path}")
                        log_entry = self.custody_logger.log_event(f"Metadata recorded for {file_path}")
                        self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

                        # Add file to archive
                        if self.archive_manager.add_to_archive(file_path):
                            print(f"  Added to archive: {file_path}")
                            log_entry = self.custody_logger.log_event(f"File archived: {file_path}")
                            self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
                    log_entry = self.custody_logger.log_event(f"Error processing file {file_path}: {e}")
                    self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

        end_timestamp = datetime.now()
        duration = end_timestamp - self.start_timestamp

        # Log agent completion event
        log_entry = self.custody_logger.log_event(f"Forensic agent scan completed in {duration}. Archive: {self.config.ARCHIVE_PATH}, Database: {self.config.DB_PATH}")
        self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
        print("Forensic scan completed.")

    def get_summary(self):
        print("--- Forensic Scan Summary ---")
        evidence_count = len(self.db_manager.get_all_evidence())
        log_count = len(self.db_manager.get_all_custody_logs())
        print(f"Total evidence files recorded: {evidence_count}")
        print(f"Total custody log entries: {log_count}")
        print(f"Evidence database: {self.config.DB_PATH}")
        print(f"Evidence archive: {self.config.ARCHIVE_PATH}")
        print("-----------------------------")

### Explanation of `ForensicAgent` Class:

*   **`__init__(self, config: 'Config')`**: The constructor takes an instance of the `Config` class, which holds all the global settings. It then initializes instances of all other component classes (`DirectoryScanner`, `FileIdentifier`, `Hasher`, `EntropyCalculator`, `DatabaseManager`, `ArchiveManager`, `CustodyLogger`, `AccessControl`). All these component classes are now defined in the consolidated code block (`cb107983`).
    *   It also records the `start_timestamp` and logs an initial event to the custody log, marking the beginning of the forensic process.
*   **`run_scan(self)`**: This is the main method that orchestrates the entire forensic investigation:
    1.  **Scanning**: It iterates through files yielded by `self.scanner.bfs_scan()`.
    2.  **Access Control**: For each `file_path`, it first checks `self.access_control.has_read_access()`. If access is denied, it logs the permission error using `handle_permission_error()` and records this event in the custody log before skipping to the next file.
    3.  **File Identification**: It calls `self.identifier.is_relevant()` to determine if the file matches the defined criteria (extension and magic number check).
    4.  **Metadata Collection & Processing (for relevant files)**:
        *   It retrieves basic file system metadata (`file_size`, `last_modified`).
        *   It computes SHA-256 and MD5 hashes using `self.hasher.hash_file()`.
        *   It calculates Shannon entropy using `self.entropy_calculator.file_entropy()`.
        *   **Database Insertion**: It attempts to insert all collected metadata into the SQLite `evidence` table via `self.db_manager.insert_evidence()`. If successful, it logs this event.
        *   **Archiving**: If metadata insertion is successful, the file is added to the `.tar.gz` archive using `self.archive_manager.add_to_archive()`, and this action is also logged.
    5.  **Error Handling**: A general `try-except` block wraps the processing of each file to catch any unforeseen errors, logging them to the custody log and allowing the scan to continue.
    6.  **Completion**: After scanning all directories, it calculates the total duration and logs a final completion event, summarizing where the evidence and logs are stored.
*   **`get_summary(self)`**: This utility method provides a brief summary of the completed scan, including the number of evidence files recorded, custody log entries, and the paths to the database and archive.

In [35]:
import os
import math
import hashlib
import sqlite3
import tarfile
import hmac
from pathlib import Path
from datetime import datetime
from collections import deque

try:
    import magic  # python-magic
    MAGIC_AVAILABLE = True
except ImportError:
    MAGIC_AVAILABLE = False

# ==========================
# CONFIGURATION
# ==========================
class Config:
    ROOT = Path(".")
    DB_PATH = Path("evidence.db")
    ARCHIVE_PATH = Path("evidence_archive.tar.gz")
    CUSTODY_KEY = b"super_secret_key_change_me"
    TARGET_EXTENSIONS = {".txt", ".pdf", ".jpg", ".jpeg", ".png"}
    MIME_MAP = {
        ".txt": ["text/plain"],
        ".pdf": ["application/pdf"],
        ".jpg": ["image/jpeg"],
        ".jpeg": ["image/jpeg"],
        ".png": ["image/png"],
    }

# ==========================
# ENTROPY (SHANNON)
# ==========================
class EntropyCalculator:
    def __init__(self, sample_size: int = 1024 * 64):
        self.sample_size = sample_size

    def file_entropy(self, path: Path) -> float:
        try:
            with path.open("rb") as f:
                data = f.read(self.sample_size)
        except Exception:
            return 0.0

        if not data:
            return 0.0

        freq = [0] * 256
        for b in data:
            freq[b] += 1

        entropy = 0.0
        length = len(data)

        for count in freq:
            if count == 0:
                continue
            p = count / length
            entropy -= p * math.log2(p)

        return entropy

# ==========================
# DIRECTORY SCANNER (BFS + GENERATORS)
# ==========================
class DirectoryScanner:
    def __init__(self, root: Path):
        self.root = Path(root)

    def bfs_scan(self):
        queue = deque([self.root])
        while queue:
            current = queue.popleft()
            if not current.exists():
                continue
            for entry in current.iterdir():
                if entry.is_dir():
                    queue.append(entry)
                else:
                    yield entry

# ==========================
# FILE IDENTIFIER (EXTENSION + MAGIC NUMBER)
# ==========================
class FileIdentifier:
    def __init__(self, target_extensions, mime_map):
        self.target_extensions = target_extensions
        self.mime_map = mime_map
        if MAGIC_AVAILABLE:
            self.magic = magic.Magic(mime=True)
        else:
            self.magic = None

    def is_relevant(self, path: Path) -> bool:
        if path.suffix.lower() not in self.target_extensions:
            return False

        if self.magic:
            try:
                actual_mime = self.magic.from_file(str(path))
                expected_mimes = self.mime_map.get(path.suffix.lower(), [])
                return actual_mime in expected_mimes
            except Exception as e:
                print(f"Warning: Could not check magic number for {path}: {e}")
                return False
        else:
            return True

# ==========================
# HASHER (SHA-256 + MD5)
# ==========================
class Hasher:
    def hash_file(self, path: Path, buffer_size: int = 65536) -> dict:
        sha256_hash = hashlib.sha256()
        md5_hash = hashlib.md5()
        try:
            with open(path, "rb") as f:
                while True:
                    data = f.read(buffer_size)
                    if not data:
                        break
                    sha256_hash.update(data)
                    md5_hash.update(data)
            return {
                "sha256": sha256_hash.hexdigest(),
                "md5": md5_hash.hexdigest()
            }
        except Exception:
            return {"sha256": "ERROR", "md5": "ERROR"}

# ==========================
# SQLITE METADATA STORE
# ==========================
class DatabaseManager:
    def __init__(self, db_path: Path):
        self.db_path = db_path
        self._create_tables()

    def _create_tables(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS evidence (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                file_path TEXT NOT NULL UNIQUE,
                file_size INTEGER,
                last_modified TEXT,
                sha256_hash TEXT,
                md5_hash TEXT,
                shannon_entropy REAL,
                timestamp TEXT NOT NULL
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS custody_log (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                timestamp TEXT NOT NULL,
                event_description TEXT NOT NULL,
                hmac_signature TEXT NOT NULL
            )
        """)
        conn.commit()
        conn.close()

    def insert_evidence(self, file_path: Path, file_size: int, last_modified: datetime, sha256_hash: str, md5_hash: str, shannon_entropy: float):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        timestamp = datetime.now().isoformat()
        try:
            cursor.execute("""
                INSERT INTO evidence (file_path, file_size, last_modified, sha256_hash, md5_hash, shannon_entropy, timestamp)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (str(file_path), file_size, last_modified.isoformat(), sha256_hash, md5_hash, shannon_entropy, timestamp))
            conn.commit()
            return True
        except sqlite3.IntegrityError:
            print(f"Warning: Evidence for {file_path} already exists. Skipping insertion.")
            return False
        except Exception as e:
            print(f"Error inserting evidence for {file_path}: {e}")
            return False
        finally:
            conn.close()

    def log_custody_event(self, event_description: str, hmac_signature: str):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        timestamp = datetime.now().isoformat()
        try:
            cursor.execute("""
                INSERT INTO custody_log (timestamp, event_description, hmac_signature)
                VALUES (?, ?, ?)
            """, (timestamp, event_description, hmac_signature))
            conn.commit()
            return True
        except Exception as e:
            print(f"Error logging custody event: {e}")
            return False
        finally:
            conn.close()

    def get_all_evidence(self) -> list:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM evidence")
        records = cursor.fetchall()
        conn.close()
        return records

    def get_all_custody_logs(self) -> list:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT * FROM custody_log")
        records = cursor.fetchall()
        conn.close()
        return records

# ==========================
# ARCHIVE MANAGER (TAR/GZ)
# ==========================
class ArchiveManager:
    def __init__(self, archive_path: Path):
        self.archive_path = archive_path
        self.files_to_archive = [] # New: list to hold paths of files to be archived

    def add_to_archive(self, file_path: Path):
        """
        Adds a specified file's path to an internal list for later archiving.
        """
        self.files_to_archive.append(file_path)
        return True # Indicate that the file is marked for archiving

    def create_final_archive(self):
        """
        Creates the final TAR.GZ archive with all collected files.
        This should be called once after all files have been identified.
        """
        if not self.files_to_archive:
            print("No files to archive.")
            return False

        original_cwd = Path.cwd()
        absolute_archive_path = self.archive_path if self.archive_path.is_absolute() else (original_cwd / self.archive_path)

        try:
            with tarfile.open(absolute_archive_path, 'w:gz') as tar:
                for file_path in self.files_to_archive:
                    if file_path.exists():
                        # Use the absolute path for 'name' and compute arcname relative to original_cwd
                        # Convert to string to avoid potential Path object issues with tar.add
                        # The first argument 'name' should be an absolute path to the file on disk.
                        # 'arcname' is what the file will be named inside the archive, relative to the archive's root.
                        # Using os.path.relpath for robustness
                        tar.add(str(file_path.resolve()), arcname=os.path.relpath(str(file_path.resolve()), str(original_cwd)))
                    else:
                        print(f"Warning: File not found for archiving: {file_path}")
            print(f"Final archive created at: {absolute_archive_path}")
            return True
        except Exception as e:
            print(f"Error creating final archive {absolute_archive_path}: {e}")
            return False

    def extract_archive(self, extract_path: Path):
        try:
            with tarfile.open(self.archive_path, 'r:gz') as tar:
                tar.extractall(path=extract_path)
            print(f"Archive extracted to {extract_path}")
            return True
        except Exception as e:
            print(f"Error extracting archive: {e}")
            return False

# ==========================
# CUSTODY LOGGER (HMAC)
# ==========================
class CustodyLogger:
    def __init__(self, secret_key: bytes):
        self.secret_key = secret_key

    def create_hmac_signature(self, data: str) -> str:
        h = hmac.new(self.secret_key, data.encode('utf-8'), hashlib.sha256)
        return h.hexdigest()

    def verify_hmac_signature(self, data: str, signature: str) -> bool:
        expected_signature = self.create_hmac_signature(data)
        return hmac.compare_digest(expected_signature, signature)

    def log_event(self, event_description: str) -> dict:
        timestamp = datetime.now().isoformat()
        log_data = f"{timestamp}|{event_description}"
        signature = self.create_hmac_signature(log_data)
        return {
            "timestamp": timestamp,
            "event_description": event_description,
            "hmac_signature": signature,
            "log_data_for_verification": log_data
        }

# ==========================
# ACCESS CONTROL
# ==========================
class AccessControl:
    def __init__(self):
        pass

    def has_read_access(self, path: Path) -> bool:
        try:
            return os.access(path, os.R_OK)
        except Exception as e:
            print(f"Warning: Could not check read access for {path}: {e}")
            return False

    def handle_permission_error(self, path: Path, action: str):
        print(f"Permission Error: Cannot {action} {path}. Skipping.")

# ==========================
# FORENSIC AGENT ORCHESTRATION
# ==========================
class ForensicAgent:
    def __init__(self, config: 'Config'):
        self.config = config
        self.scanner = DirectoryScanner(config.ROOT)
        self.identifier = FileIdentifier(config.TARGET_EXTENSIONS, config.MIME_MAP)
        self.hasher = Hasher()
        self.entropy_calculator = EntropyCalculator()
        self.db_manager = DatabaseManager(config.DB_PATH)
        self.archive_manager = ArchiveManager(config.ARCHIVE_PATH)
        self.custody_logger = CustodyLogger(config.CUSTODY_KEY)
        self.access_control = AccessControl()
        self.start_timestamp = datetime.now()

        log_entry = self.custody_logger.log_event("Forensic agent initiated scan.")
        self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
        print(f"Forensic Agent initialised. Scanning from: {self.config.ROOT}")

    def run_scan(self):
        print("Starting forensic scan...")
        for file_path in self.scanner.bfs_scan():
            if not self.access_control.has_read_access(file_path):
                self.access_control.handle_permission_error(file_path, "read")
                log_entry = self.custody_logger.log_event(f"Permission denied for file: {file_path}. Skipped.")
                self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
                continue

            if self.identifier.is_relevant(file_path):
                print(f"Found relevant file: {file_path}")
                try:
                    file_stat = file_path.stat()
                    file_size = file_stat.st_size
                    last_modified = datetime.fromtimestamp(file_stat.st_mtime)

                    hashes = self.hasher.hash_file(file_path)
                    sha256_hash = hashes['sha256']
                    md5_hash = hashes['md5']

                    shannon_entropy = self.entropy_calculator.file_entropy(file_path)

                    if self.db_manager.insert_evidence(file_path, file_size, last_modified, sha256_hash, md5_hash, shannon_entropy):
                        print(f"  Metadata recorded for {file_path}")
                        log_entry = self.custody_logger.log_event(f"Metadata recorded for {file_path}")
                        self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

                        # Add file to internal list for later archiving
                        if self.archive_manager.add_to_archive(file_path):
                            print(f"  Marked for archiving: {file_path}")
                            log_entry = self.custody_logger.log_event(f"File marked for archiving: {file_path}")
                            self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
                    log_entry = self.custody_logger.log_event(f"Error processing file {file_path}: {e}")
                    self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

        end_timestamp = datetime.now()
        duration = end_timestamp - self.start_timestamp

        # Create the final archive after all files have been processed
        print("Finalizing archive...")
        if self.archive_manager.create_final_archive():
            log_entry = self.custody_logger.log_event(f"Final evidence archive created at: {self.config.ARCHIVE_PATH}")
            self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
        else:
            log_entry = self.custody_logger.log_event(f"Failed to create final evidence archive.")
            self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])

        # Log agent completion event
        log_entry = self.custody_logger.log_event(f"Forensic agent scan completed in {duration}. Archive: {self.config.ARCHIVE_PATH}, Database: {self.config.DB_PATH}")
        self.db_manager.log_custody_event(log_entry['event_description'], log_entry['hmac_signature'])
        print("Forensic scan completed.")

    def get_summary(self):
        print("--- Forensic Scan Summary ---")
        evidence_count = len(self.db_manager.get_all_evidence())
        log_count = len(self.db_manager.get_all_custody_logs())
        print(f"Total evidence files recorded: {evidence_count}")
        print(f"Total custody log entries: {log_count}")
        print(f"Evidence database: {self.config.DB_PATH}")
        print(f"Evidence archive: {self.config.ARCHIVE_PATH}")
        print("-----------------------------")

### Initializing and Running the `ForensicAgent`

To see the `ForensicAgent` in action, we'll first ensure all our component classes are defined (which was done in the consolidated code block `cb107983`), then we'll instantiate the `Config` object, followed by the `ForensicAgent`, trigger the scan, and finally retrieve a summary of its findings.

In [19]:
import os
from pathlib import Path

# Instantiate the Config object (defined in the consolidated code block)
config = Config()

# Instantiate the ForensicAgent
forensic_agent = ForensicAgent(config)

# Run the scan
forensic_agent.run_scan()

# Get a summary of the scan results
forensic_agent.get_summary()

Forensic Agent initialised. Scanning from: .
Starting forensic scan...
Found relevant file: Executable codes Assig 2.txt
  Metadata recorded for Executable codes Assig 2.txt
Error adding Executable codes Assig 2.txt to archive: mode must be 'r', 'w' or 'x'
Forensic scan completed.
--- Forensic Scan Summary ---
Total evidence files recorded: 1
Total custody log entries: 3
Evidence database: evidence.db
Evidence archive: evidence_archive.tar.gz
-----------------------------


In [9]:
import math
from pathlib import Path

class EntropyCalculator:
    def __init__(self, sample_size: int = 1024 * 64):
        # sample_size limits how much of the file we read for entropy estimation
        self.sample_size = sample_size

    def file_entropy(self, path: Path) -> float:
        """
        Estimate Shannon entropy (in bits per byte) for the given file.
        High values may indicate encryption or compression.
        """
        try:
            with path.open("rb") as f:
                data = f.read(self.sample_size)
        except Exception:
            return 0.0

        if not data:
            return 0.0

        freq = [0] * 256
        for b in data:
            freq[b] += 1

        entropy = 0.0
        length = len(data)

        for count in freq:
            if count == 0:
                continue
            p = count / length
            entropy -= p * math.log2(p)

        return entropy

### Explanation of `FileIdentifier` Class:

*   **`try...except ImportError` for `magic`**: This block attempts to import the `python-magic` library. If the library is not installed (which might be the case in some environments), it gracefully sets `MAGIC_AVAILABLE` to `False`. This allows the agent to function, albeit with a less robust identification method, if `python-magic` isn't present.
*   **`__init__(self, target_extensions, mime_map)`**: The constructor takes two arguments:
    *   `target_extensions`: A set of file extensions that are considered relevant (e.g., `{'.txt', '.pdf'}`). This comes from the `Config`.
    *   `mime_map`: A dictionary mapping extensions to a list of expected MIME types (also from `Config`).
    *   It initializes `self.magic` with `magic.Magic(mime=True)` if `python-magic` is available. This object is used to determine the MIME type of a file based on its content.
*   **`is_relevant(self, path: Path) -> bool`**: This is the core method for file identification.
    1.  **Extension Filter**: It first checks if the file's extension (`path.suffix.lower()`) is present in the `self.target_extensions` set. If not, the file is immediately deemed not relevant.
    2.  **Magic Number / MIME Verification**: If `python-magic` is available (`self.magic` is not `None`):
        *   It uses `self.magic.from_file(str(path))` to read the file's content and determine its actual MIME type.
        *   It then compares this `actual_mime` type against the `expected_mimes` defined in `self.mime_map` for that specific file extension.
        *   If the actual MIME type is not among the expected types, the file is considered not relevant, even if its extension matched. This prevents malicious files from being disguised by simply renaming them.
        *   A `try-except` block is included to handle potential errors during the `magic.from_file` call (e.g., if the file is corrupted or permissions prevent reading).
    3.  **Fallback**: If `python-magic` is *not* available, the method relies solely on the extension filter and returns `True` if the extension matches. This ensures the agent can still perform basic identification.

### Explanation of Module Structure:

*   **`forensic_agent/`**: This is the root directory for your project.
*   **`config.py`**: Would contain configuration settings like file paths, secret keys, etc.
*   **`scanner.py`**: Would handle the directory scanning logic (like the BFS).
*   **`identifier.py`**: Would contain logic for identifying relevant files (extension, magic number).
*   **`hasher.py`**: Would implement hashing functions (SHA-256, MD5).
*   **`entropy.py`**: Would contain the Shannon entropy calculation logic.
*   **`db.py`**: Would manage the SQLite database interactions.
*   **`archiver.py`**: Would handle archiving evidence files.
*   **`custody.py`**: Would manage custody logging with HMAC signatures.
*   **`access_control.py`**: Would implement permission handling.
*   **`agent.py`**: This would be the main orchestration file, bringing all the other modules together to run the forensics agent.
*   **`tests/`**: A directory to hold unit tests for each component, ensuring everything works as expected.

This modular approach makes the code more organized, reusable, and easier to debug, but for a single Colab notebook, all this code is typically combined into one file, as we will see next.

## 2. Full Executable Code (Single file for Google Colab)

Below is the complete, self-contained Python code designed to run directly in a single Google Colab cell (cell `cb107983`). It consolidates all the class definitions previously discussed. We'll go through its structure section by section.

### 2.1. CONFIGURATION

This section defines a `Config` class that holds all the essential settings for our forensic agent. Think of this as the control panel where you can adjust how the agent behaves. It imports `os` for operating system interaction and `pathlib.Path` for working with file paths in a more object-oriented way.

In [ ]:
import os
from pathlib import Path

class Config:
    # Root directory to scan (change this in Colab as needed)
    ROOT = Path(".")  # current directory

    # SQLite database path
    DB_PATH = Path("evidence.db")

    # Archive path
    ARCHIVE_PATH = Path("evidence_archive.tar.gz")

    # Custody secret key (for HMAC signing)
    CUSTODY_KEY = b"super_secret_key_change_me"

    # Target extensions
    TARGET_EXTENSIONS = {".txt", ".pdf", ".jpg", ".jpeg", ".png"}

    # Mapping from extension to expected MIME types
    MIME_MAP = {
        ".txt": ["text/plain"],
        ".pdf": ["application/pdf"],
        ".jpg": ["image/jpeg"],
        ".jpeg": ["image/jpeg"],
        ".png": ["image/png"],
    }

config = Config()

### Explanation of `Config` Class:

*   **`ROOT = Path('.')`**: This sets the starting directory for the scan. `.` means the current directory where the script is being run. You can change this to `/content` or any other path if you want to scan a specific folder in Colab.
*   **`DB_PATH = Path('evidence.db')`**: This defines the name and path for the SQLite database file where all the collected metadata will be stored.
*   **`ARCHIVE_PATH = Path('evidence_archive.tar.gz')`**: This specifies the name and path for the compressed archive file that will contain the collected evidence.
*   **`CUSTODY_KEY = b"super_secret_key_change_me"`**: This is a secret key used for signing audit log entries with HMAC (Hash-based Message Authentication Code). It ensures the integrity and authenticity of the logs. **It's crucial to change this to a strong, unique secret key in a real-world scenario!** The `b` prefix means it's a byte string.
*   **`TARGET_EXTENSIONS = {'.txt', '.pdf', ...}`**: This is a set of file extensions that the agent will consider 'relevant' for forensic analysis. Only files with these extensions will be further processed.
*   **`MIME_MAP = {...}`**: This is a dictionary that maps file extensions to their expected MIME (Multipurpose Internet Mail Extensions) types. MIME types are standard identifiers for file formats (e.g., `text/plain` for text files, `image/jpeg` for JPG images). This map is used in conjunction with magic number checks to confirm the true nature of a file, preventing attackers from disguising malicious files by simply changing their extension.

Finally, `config = Config()` creates an instance of this `Config` class, making all these settings easily accessible throughout the agent's code.